In [ ]:
from glob import glob
from cmcmultimodal.io import mat2nii
from pathlib import Path
import os

my_files = glob('/Users/Vasilis/Downloads/Zebel/PSOCT/Retardance/Slice*.mat')
highres_files = []
lowres_files = []
for f in my_files:
    nii_highres, nii_lowres = mat2nii(f, nii_lowres_file=os.path.join(Path(f).parent,'lowres/',
                                      Path(f).name.replace('.mat','.nii.gz')), downsample=10)
    highres_files.append(nii_highres)
    lowres_files.append(nii_lowres)
    
highres_files

In [ ]:
# Process all MOE slides

from cmcmultimodal.proc import psoct
from pathlib import Path

datadir = '/Users/Vasilis/Downloads/PSOCT_Ret'
output_path = '/Users/Vasilis/Downloads/PSOCT_all_slides_Mixed_2'
mri_ref = '/Users/Vasilis/Downloads/PSOCT_Ret/MRI/reoriented_FA.nii.gz'
seq_params = '/Users/Vasilis/Downloads/PSOCT_Ret/PSOCT/seq_params.json'

# my_data = psoct(Path(datadir), seq_params, slide_range=(98,200), reg_modality='Cross', verbose=True)
my_data = psoct(Path(datadir), seq_params, slide_range=(160,200), reg_modality='Mixed', verbose=True)

indiv_slides = my_data.run_pipeline(other_images=['Retardance'], 
                                    output_path=output_path, 
                                    mri_ref=mri_ref, 
                                    downsample=1, 
                                    bad_slides=[140,])

In [28]:
import json
import numpy as np
from pathlib import Path

output_path = Path('/Users/Vasilis/Downloads/PSOCT_all_slides_Cross_7')

with open(output_path / "abs_shifts.json") as f:
    data = json.load(f)

abs_shifts = {int(k): np.array(v) for k, v in data.items()}

In [17]:
from fsl.wrappers import flirt
from cmcmultimodal.utils import pad_image, save_nifti
from fsl.data.image import Image
import numpy as np

out_path=Path('/Users/Vasilis/Downloads/PSOCT_all_slides_Mixed_2/Retardance')
inp_path=Path('/Users/Vasilis/Downloads/PSOCT_Ret/PSOCT/Retardance')
ref_path=Path('/Users/Vasilis/Downloads/PSOCT_all_slides_Mixed_2/Mixed/lowres')

i = 180

ref = ref_path / ('Slice_' + str(i) + '_EnRCr_hdr')
src = inp_path / ('Slice_' + str(i) + '_EnR')

img_highres = Image(src).data[:,:,0]
highres_shape = [x * my_data.downsample for x in my_data.ref_shape]
img_padded    = pad_image(img_highres, highres_shape)
src_filename  = 'tmp_padded_source.nii.gz'
save_nifti(img_padded, src_filename)

mat = np.loadtxt(out_path / '../slides_to_mri.mat')

T = mat * my_data.abs_shifts[i]

img_padded = flirt(src_filename, ref, init=T, out=out_path / (src.name + '_hdr'), twod=True, applyxfm=True)


In [14]:
highres_shape

[11000, 8740]

In [15]:
img_padded

{}

In [ ]:
from fsl.wrappers import applyxfm, flirt
from cmcmultimodal.utils import pad_image
from fsl.data.image import Image
from cmcmultimodal.io import save_nifti

out_path=Path('/Users/Vasilis/Downloads/PSOCT_test/')
inp_path=Path('/Users/Vasilis/Downloads/PSOCT_Ret/PSOCT/')
ref_path=Path('/Users/Vasilis/Downloads/PSOCT_all_slides_Cross_7')

# i = 150
for i in abs_shifts.keys():
    inp = inp_path / 'Retardance' / 'lowres' / ('Slice_' + str(i).zfill(3) +'_EnR.nii.gz')
    # inp = inp_path / 'Cross' / 'lowres' / ('Slice_' + str(i) +'_EnCr.nii.gz')
    ref = ref_path / 'Cross' / 'lowres' / 'Slice_184_EnCr_hdr.nii.gz'
    try:
        inp_image = Image(inp).data[:,:,0]
    except:
        continue
    # shape = Image(ref).data[:,0,:].shape
    shape=(1100, 874)

    src_padded = pad_image(inp_image, shape)
    src_filename = out_path / 'padded_source'
    save_nifti(src_padded, src_filename)

    # applyxfm(src_filename, ref, abs_shifts[i], out_path / inp.name, twod=True)
    flirt(src_filename, ref, init=abs_shifts[i], out=out_path / inp.name, twod=True, applyxfm=True)



In [ ]:
from fsl.wrappers import flirt, LOAD, applyxfm
from cmcmultimodal.utils import pad_image
from cmcmultimodal.io import save_nifti
import numpy as np
import os
from fsl.data.image         import Image

# inp_path='/Users/Vasilis/Downloads/PSOCT_Ret/PSOCT/Retardance/lowres/'
inp_path='/Users/Vasilis/Downloads/PSOCT_Ret/PSOCT/Cross/lowres/'
out_path='/Users/Vasilis/Downloads/PSOCT_test/'
os.makedirs(out_path, exist_ok=True)
for i in [160]:#range(98,201):
    # inp='Slice_' + str(i) +'_EnR.nii.gz'
    # ref='Slice_325_EnR.nii.gz'
    inp='Slice_' + str(i) +'_EnCr.nii.gz'
    ref='Slice_161_EnCr.nii.gz'
    # shape=(1100, 874)
    # # shape = [i+min(shape)//10 for i in shape]
    try:
        src_data = Image(inp_path+inp).data[:,:,0]
    except:
        continue
    tgt_data = Image(inp_path+ref).data[:,:,0]
    shape = tuple(map(max, zip(src_data.shape, tgt_data.shape)))
    # increase max shape by 10% each side
    shape = tuple(i+min(shape)//5 for i in shape)
    src_padded = pad_image(src_data, shape)
    tgt_padded = pad_image(tgt_data, shape)
    src_filename = out_path + 'padded_source'
    tgt_filename = out_path + 'padded_target'
    save_nifti(src_padded, src_filename)
    save_nifti(tgt_padded, tgt_filename)
    t = flirt(src_filename, tgt_filename,
                omat=LOAD,# out= out_path + inp,
                cost='leastsq',
                twod=True)
    # cost='normcorr' or 'leastsq'
    # t= flirt(src_filename, tgt_filename,
    #      omat='tmpmat1.mat',
    #      out='tmpfile1', cost='leastsq',
    #      twod=True)
    # t = np.loadtxt('tmpmat1.mat')
    print(t['omat'])
    img_padded = applyxfm(src_filename, tgt_filename, t['omat'], out_path + inp, twod=True)
    # img_padded = flirt(src_filename, tgt_filename, init=t['omat'], out=out_path + 'Try2_' + inp, applyxfm=True,
                # )
    # img_padded['out'].get_fdata()
# print(img_padded)
# flirt(src_filename, tgt_filename,
#      omat='tmpmat2.mat',
#      out='tmpfile2', cost='normcorr',
#      twod=True)
# t = np.loadtxt('tmpmat2.mat')
# print(t)

[[ 1.00000000e+00  1.89414713e-05  0.00000000e+00  4.53207593e-01]
 [-1.89414713e-05  1.00000000e+00  0.00000000e+00  1.47949071e+00]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


In [ ]:
from fsl.data.image         import Image
import numpy as np
from pathlib import Path

ref_path = Path('/Users/Vasilis/Downloads/PSOCT_test_output_5')
est_path = Path('/Users/Vasilis/Downloads/PSOCT_test_output_6')

def compare_images(ref, est):
    ref_img = Image(ref)
    est_img = Image(est)

    if not np.allclose(ref_img.data, est_img.data):
        print('\t', 'Images are NOT equal!')
    if ref_img.header != est_img.header:
        print('\t', 'Headers are NOT equal!')

def compare_matrices(ref, est):
    ref_mat = np.loadtxt(ref)
    est_mat = np.loadtxt(est)

    if not np.allclose(ref_mat, est_mat, atol=0.001):
        print('\t', 'Matrices are NOT equal!')

for file in ref_path.glob('*'):
    if file.is_dir():
        for subfile in file.glob('*'):
            print(subfile)
            corresponding_est_file = est_path / file.name / subfile.name
            if corresponding_est_file.exists() == False:
                print('\t', 'File does not exist in estimated path!')
                continue
            if subfile.suffix in ['.nii', '.gz']:
                compare_images(subfile, corresponding_est_file)
            elif subfile.suffix == '.mat':
                compare_matrices(subfile, corresponding_est_file)
    else:
        print(file)
        corresponding_est_file = est_path / file.name
        if corresponding_est_file.exists() == False:
            print('\t', 'File does not exist in estimated path!')
            continue
        if file.suffix in ['.nii', '.gz']:
            compare_images(file, corresponding_est_file)
        elif file.suffix == '.mat':
            compare_matrices(file, corresponding_est_file)


In [10]:
# test fslsplit in the three orientations
from fsl.wrappers.avwutils  import fslsplit

inp_path = Path('/Users/Vasilis/Downloads/PSOCT_test_output_3')
out_path = Path('/Users/Vasilis/Downloads/fslsplit_test')
slide_deck = inp_path / 'slide_deck.nii.gz'
orientation = 'sagittal'  

# Lookup table for orientation information
OrientationLookup = {'sagittal': 'x', 'coronal': 'y', 'axial': 'z',
                     'Sagittal': 'x', 'Coronal': 'y', 'Axial': 'z'}

indiv_slides = fslsplit(src=slide_deck, out=out_path/orientation, dim=OrientationLookup[orientation])
